# Recursive Contracts and Divide-and-Conquer


```{contents}
:local:
:depth: 2
```


```{index} Let a Helper Promise Exactly One Smaller Problem
```



```{index} Let a Helper Promise Exactly One Smaller Problem
```

## Let a Helper Promise Exactly One Smaller Problem



A public method can accept a complete report while a private recursive helper accepts a suffix or range. The helper contract must name that smaller problem precisely. For a suffix starting at index i, the result might be the total of entries i through the end. The public caller starts with i zero; other starting indexes are an implementation detail rather than a burden placed on every client.

The base case is an empty suffix, not a special final value guessed from one example. Its sum is zero. For a nonempty suffix, combine the first available entry with the helper's result on the remaining suffix. The progress measure n-i decreases by one, which connects the contract to termination. The caller still needs an arithmetic-range policy: termination does not prove that a fixed-width sum cannot overflow.

For divide-and-conquer, represent a range as [lo, hi). Its length is hi-lo, and empty means lo equals hi. Splitting at lo+(hi-lo)/2 gives two adjacent half-open ranges. On five items, the lengths are two and three; neither item duplication nor equal lengths are required. What matters is complete, nonoverlapping coverage and strictly smaller children when recursion continues.

| Design question | What to state before recursing |
| --- | --- |
| What does the helper return? | A result for this suffix/range |
| When is the answer direct? | Empty/singleton or another explicit base |
| What becomes smaller? | Remaining length or another well-founded measure |
| How are child results combined? | An operation justified by the parent contract |

Guided contrast: finding a maximum needs results from both halves, while a binary-search comparison chooses only one remaining half. Both use interval splitting, but they do not have the same call tree or work recurrence. Predict child calls for one size-five input under each algorithm before comparing their costs.

For related explanation, see the further reading below.[^development-reading]


```{index} Recursive Problem Solving
```

<span id="recursive-problem-solving"></span>


Basic recursive cases, base answers, and call/return tracing are taught in {doc}`Chapter 17 <../17_trees/1701_tree_structure>`. Review that section before this optional extension. Here you develop helper contracts, progress arguments, resource limits, and divide-and-conquer design.


```{index} Progress Measures
```

## Progress Measures




```{index} Progress Measures
```

A progress measure proves that recursion is moving in the right direction. The measure does not have to be a single number, but it must get closer to a base case.

In [ ]:
using System;
using System.Linq;

int[] values={2,4,6,8,10};
Console.WriteLine(Contains(values,6));Console.WriteLine(Contains(values,7));
Console.WriteLine(Contains(Array.Empty<int>(),6));
bool Contains(int[] values,int target)
{
 ArgumentNullException.ThrowIfNull(values);
 return Scan(0);
 bool Scan(int index)
 {
  if(index==values.Length)return false;
  if(values[index]==target)return true;
  return Scan(index+1);
 }
}

True
False
False


The wrapper hides the helper index so clients cannot supply a negative or out-of-range start. Scan(index) promises whether the target occurs in the suffix [index,n). An empty suffix returns false; a matching first item returns true; otherwise the answer comes from the smaller suffix. The measure n−index decreases on every recursive call. Empty input needs no array access. Worst-case time and stack are Θ(n); a first-item match needs only one helper frame.

**Common mistake.** Letting the helper take the whole array as its parameter and recursing on a copy without the first element. Each call then allocates a new array, so the search costs Θ(n²) copying for a Θ(n) scan; an index into the shared array is the progress measure that costs nothing to advance.


```{index} Forward and Backward Flow
```

### Forward and Backward Flow

Every recursive method has two moments where it can do work: *before* its recursive call, while the calls are still being made, and *after* the call returns, while they are unwinding. Work done before the call runs in forward order, first call first; work done after the call runs in backward order, deepest call first. The same method body with the print statement moved from one side of the call to the other reverses the output, which is the whole trick behind many recursive solutions.

A useful picture is a row of dominoes. Pushing the first one is the first call; each domino knocks the next, which is the chain of calls down to the base case; and if each domino also reports back when the one after it has fallen, those reports arrive in reverse order. Printing an invoice list in arrival order is forward flow. Printing it newest first, converting a number to binary (the last digit computed is the first one printed), or reversing a string are backward flow: the answer for the whole is built after the answer for the rest is known. The example runs both flows over the same list of order numbers.

In [ ]:
using System;

int[] orders = { 101, 102, 103, 104 };
Console.Write("forward:  "); Forward(0); Console.WriteLine();
Console.Write("backward: "); Backward(0); Console.WriteLine();
Console.WriteLine($"binary of 45: {ToBinary(45)}");

void Forward(int i)
{
    if (i == orders.Length) return;
    Console.Write(orders[i] + " ");      // work before the call: forward order
    Forward(i + 1);
}

void Backward(int i)
{
    if (i == orders.Length) return;
    Backward(i + 1);
    Console.Write(orders[i] + " ");      // work after the call: reverse order
}

string ToBinary(int n) => n < 2 ? n.ToString() : ToBinary(n / 2) + (n % 2);   // the last digit found is appended last

forward:  101 102 103 104 
backward: 104 103 102 101 
binary of 45: 101101


The two methods differ only in the position of one line, and they print `101 102 103 104` and `104 103 102 101`. In `Backward`, every call waits for the rest of the list to print before printing its own element, so the deepest call, which holds the last order, prints first. `ToBinary` is backward flow with a return value: the remainder `n % 2` is the *last* binary digit of n, so it must come after the digits of `n / 2`, which the recursive call supplies; 45 is 101101 because 45 / 2 = 22 remainder 1, 22 / 2 = 11 remainder 0, and so on up to 1. Writing the concatenation the other way round, `(n % 2) + ToBinary(n / 2)`, compiles and reverses the digits.

```{index} Code Completion: Sum of the Digits
```

### Code Completion: Sum of the Digits

Complete `DigitSum` so that it returns the sum of the decimal digits of a nonnegative integer. Supply the base case and the recursive case; the smaller problem is the number with its last digit removed, `n / 10`, and the work at this level is the last digit, `n % 10`. The expected output is `18`, `7`, `0`.

In [ ]:
using System;

Console.WriteLine(DigitSum(1980));
Console.WriteLine(DigitSum(7));
Console.WriteLine(DigitSum(0));

static int DigitSum(int n)
{
    // Your code starts here.
    throw new NotImplementedException();
    // Your code ends here.
}

In [ ]:
// Solution
using System;

Console.WriteLine(DigitSum(1980));
Console.WriteLine(DigitSum(7));
Console.WriteLine(DigitSum(0));

static int DigitSum(int n)
{
    if (n < 10) return n;                       // a single digit is its own sum
    return n % 10 + DigitSum(n / 10);           // last digit plus the sum of the rest
}

18
7
0


The base case `n < 10` covers 0 through 9 in one line; a base case of `n == 0` alone also works but makes every call go one level deeper. The progress measure is the number of digits, which drops by one at each call because integer division by 10 removes the last digit. A common wrong attempt, `n % 10 + DigitSum(n % 10)`, never shrinks n and recurses until the stack is exhausted; check that the recursive argument is strictly smaller before running.

```{index} Code Tracing: Find the Error
```

### Code Tracing: Find the Error

Each method below is meant to compute what its name says, and each has one defect. Without running them, decide what each actually does for the call shown: wrong answer, exception, or endless recursion. Then open the solution, which runs the two that terminate and explains the third.

```csharp
static int CountDigits(int n) => n == 0 ? 0 : 1 + CountDigits(n / 10);          // CountDigits(0)
static long Power(int b, int e) => e == 0 ? 1 : b * Power(b, e - 1) * b;          // Power(2, 3)
static int Largest(int[] a, int i) => i == a.Length ? a[i] : Math.Max(a[i], Largest(a, i + 1));   // Largest(new[]{4,9,2}, 0)
```

In [ ]:
// Solution
using System;

Console.WriteLine($"CountDigits(0) = {CountDigits(0)}  (expected 1: zero has one digit; the base case returns 0)");
Console.WriteLine($"Power(2, 3) = {Power(2, 3)}  (expected 8: each level multiplies by b twice, so the result is b to the 2e)");
try { Console.WriteLine(Largest(new[] { 4, 9, 2 }, 0)); }
catch (IndexOutOfRangeException) { Console.WriteLine("Largest throws IndexOutOfRangeException: the base case reads a[a.Length]"); }

static int CountDigits(int n) => n == 0 ? 0 : 1 + CountDigits(n / 10);
static long Power(int b, int e) => e == 0 ? 1 : b * Power(b, e - 1) * b;
static int Largest(int[] a, int i) => i == a.Length ? a[i] : Math.Max(a[i], Largest(a, i + 1));

CountDigits(0) = 0  (expected 1: zero has one digit; the base case returns 0)
Power(2, 3) = 64  (expected 8: each level multiplies by b twice, so the result is b to the 2e)
Largest throws IndexOutOfRangeException: the base case reads a[a.Length]


None of the three recurses forever; each terminates and is wrong in a different way. `CountDigits` has a base case that is correct for the recursion but wrong for the input 0 itself, which should count as one digit; the fix is a base case `n < 10 ? 1`. `Power` does the right number of calls but doubles the work at each level, returning 64 instead of 8. `Largest` reads one past the end in its base case; the base case for a nonempty suffix of length one is `i == a.Length - 1 ? a[i]`, with the empty array rejected by the public wrapper. Finding the defect by reading the base case first is the habit this exercise builds.

```{index} Recursion and Iteration
```

## Recursion and Iteration




```{index} Recursion and Iteration
```

Many recursive methods can be written with loops. Use recursion when the problem naturally breaks into smaller problems of the same shape, or when recursion makes the structure easier to see.

In [ ]:
using System;
using System.Linq;

int[] values={10,-3,8};
Console.WriteLine($"recursive={Sum(values)}, iterative={values.Sum(x=>(long)x)}");
long Sum(int[] values)
{
 ArgumentNullException.ThrowIfNull(values);
 return From(0);
 long From(int index)=>index==values.Length?0:values[index]+From(index+1);
}

recursive=15, iterative=15


Both sums visit every item, taking Θ(n) time under the integer cost model. The recursive version retains Θ(n) active frames; an iterative sum retains Θ(1) auxiliary state. Recursion changes expression of control flow, not the amount of data that must be processed. Prefer the loop for long flat reports; recursion is particularly useful for balanced splits and nested structures.

**Common mistake.** Using recursion for a long flat report because it "reads better". Each item adds a stack frame, so a list of a few hundred thousand amounts can exhaust the runtime stack where the loop needs one variable; recursion earns its cost on nested or halving structures, not on flat sequences.


```{index} Recursive Risks
```

## Recursive Risks




```{index} Recursive Risks
```

A recursive method with a missing base case or no progress can call itself forever until the program runs out of stack space. In C#, recursive calls are not something to use casually for huge linear loops.

In [ ]:
using System;

Console.WriteLine(Countdown(3));
foreach(int n in new[]{-1,101})
{
 try{Countdown(n);}catch(ArgumentOutOfRangeException){Console.WriteLine($"reject {n}");}
}
int Countdown(int n)
{
 if(n<0||n>100)throw new ArgumentOutOfRangeException(nameof(n),"Small teaching input required.");
 return n==0?0:1+Countdown(n-1);
}


3
reject -1
reject 101


Check input domain, direct base answer, and a well-founded progress measure before running recursive code. Do not intentionally trigger stack exhaustion in the browser. Tail-call elimination is not guaranteed for these C# methods. A finite domain and arithmetic range are separate from the proof of termination; validate both when designing the interface.

**Common mistake.** Relying on the exception to stop a runaway recursion. A `StackOverflowException` cannot be caught and ends the process; the domain check in `Countdown` runs *before* the first recursive call, which is the only place a guard can protect the stack.


```{index} Exercise
```

## Exercise




```{index} Exercise
```

Recursively count positive expense adjustments in a suffix. Zero is not positive. State the helper postcondition and progress measure; test empty input.

In [ ]:
using System;
using System.Linq;

int[] values={-2,0,5,7};
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;
using System.Linq;

// Solution
int[] values={-2,0,5,7};
Console.WriteLine($"positive={Count(values)}, empty={Count(Array.Empty<int>())}");
int Count(int[] values)
{
 return From(0);
 // From(i) counts positive values in [i,n); measure n-i.
 int From(int i)=>i==values.Length?0:(values[i]>0?1:0)+From(i+1);
}

positive=2, empty=0


The output is `positive=2, empty=0`. The 0 was not counted, because the test is `> 0`, and the empty array returned 0 from the base case without reading any element.


```{index} Divide-and-Conquer Design
```

<span id="divide-and-conquer-design"></span>


Divide-and-conquer algorithms solve a problem in three phases: divide the input into smaller parts, solve the smaller parts, and combine their results. Recursion is a natural implementation technique because each subproblem has the same shape as the original problem.


```{index} The Pattern
```

## The Pattern




```{index} The Pattern
```

A divide-and-conquer method usually follows this shape:

1. If the problem is small enough, solve it directly.
2. Divide the input into smaller pieces.
3. Recursively solve the pieces.
4. Combine the partial answers.

In [ ]:
using System;
using System.Linq;

foreach(int[] values in new[]{new[]{3,9,2,7,5},new[]{-8},Array.Empty<int>()})
{
 try{Console.WriteLine($"maximum={Max(values)}");}
 catch(ArgumentException){Console.WriteLine("empty maximum rejected");}
}
int Max(int[] values)
{
 ArgumentNullException.ThrowIfNull(values);
 if(values.Length==0)throw new ArgumentException("Nonempty input required.");
 return Solve(0,values.Length);
 int Solve(int start,int end)
 {
  if(end-start==1)return values[start];
  int middle=start+(end-start)/2;
  return Math.Max(Solve(start,middle),Solve(middle,end));
 }
}

maximum=9
maximum=-8
empty maximum rejected


The helper promise is the maximum of the nonempty half-open range [start,end). The wrapper rejects empty input before recursion. For length>1, both child ranges are nonempty, smaller, disjoint, and together cover the parent, including odd lengths. Inductively correct child maxima combine to the parent maximum. The size measure end−start decreases. T(n)=T(floor(n/2))+T(ceil(n/2))+Θ(1), hence Θ(n) work and Θ(log n) stack. A loop also takes Θ(n) work with less stack: splitting alone does not make maximum selection faster.

**Common mistake.** Splitting as `Solve(start, middle)` and `Solve(middle + 1, end)`. The element at `middle` belongs to neither half, so one value is silently skipped; with half-open ranges the second call starts exactly at `middle`.


```{index} Two Recursive Calls: Towers of Hanoi
```

### Two Recursive Calls: Towers of Hanoi

Divide-and-conquer sometimes solves two subproblems that are *not* halves. The classic example is moving a stack of n disks from one peg to another using a spare peg, when only one disk moves at a time and a larger disk may never sit on a smaller one. Think of it as relocating a pile of n archive boxes of decreasing size with the same stacking rule. The recursive insight is to state the subproblem exactly: to move n boxes from the source to the target, move the top n - 1 boxes to the spare, move the largest box to the target, then move the n - 1 boxes from the spare onto it. Each of the two recursive calls solves the same problem for n - 1 with the pegs relabeled.

The cost recurrence is M(1) = 1 and M(n) = 2M(n - 1) + 1, which expands to 2ⁿ - 1 moves. The number of moves doubles with each additional box, so the method is correct for any n and useless for large n; 64 boxes would need about 1.8 × 10¹⁹ moves. That is the recurrence of 21.3's concept check in action: two calls on a problem smaller by one, not by half.

In [ ]:
using System;

int moves = 0;
Move(3, "A", "C", "B");
Console.WriteLine($"3 boxes: {moves} moves");
foreach (int n in new[] { 1, 5, 10, 20 })
{
    moves = 0;
    Move(n, "A", "C", "B", print: false);
    Console.WriteLine($"{n,2} boxes: {moves,9} moves = 2^{n} - 1");
}

void Move(int n, string from, string to, string spare, bool print = true)
{
    if (n == 0) return;
    Move(n - 1, from, spare, to, print);      // clear the way: n - 1 boxes to the spare peg
    moves++;
    if (print) Console.WriteLine($"  move box {n} from {from} to {to}");
    Move(n - 1, spare, to, from, print);      // bring them back on top of the big box
}

  move box 1 from A to C
  move box 2 from A to B
  move box 1 from C to B
  move box 3 from A to C
  move box 1 from B to A
  move box 2 from B to C
  move box 1 from A to C
3 boxes: 7 moves
 1 boxes:         1 moves = 2^1 - 1
 5 boxes:        31 moves = 2^5 - 1
10 boxes:      1023 moves = 2^10 - 1
20 boxes:   1048575 moves = 2^20 - 1


Three boxes take seven moves, and the printed sequence never puts a larger box on a smaller one, although no line of the method checks that rule: it holds because each recursive call is entered only when the boxes it will move are the top n - 1 and the peg it uses is free of smaller boxes. The counts 1, 31, 1,023, and 1,048,575 confirm 2ⁿ - 1. Trace the n = 3 output and label each line with the call that produced it; the first three moves come from the first recursive call, the fourth from the middle line, and the last three from the second call, the same shape as the recursion tree in 16.4.

```{index} Code Tracing: How Many Calls?
```

### Code Tracing: How Many Calls?

For each call below, predict how many times the method is entered in total, counting the original call, and what it returns. Then open the solution.

```csharp
static int Pairs(int n) => n <= 1 ? 1 : Pairs(n - 1) + Pairs(n - 2);     // Pairs(5)
static int Halve(int n) => n == 0 ? 0 : 1 + Halve(n / 2);                // Halve(40)
static int Spread(int n) => n <= 1 ? 1 : Spread(n / 2) + Spread(n / 2);  // Spread(8)
```

In [ ]:
// Solution
using System;

int calls = 0;
calls = 0; int p = Pairs(5);  Console.WriteLine($"Pairs(5) = {p}, entered {calls} times");
calls = 0; int h = Halve(40); Console.WriteLine($"Halve(40) = {h}, entered {calls} times");
calls = 0; int s = Spread(8); Console.WriteLine($"Spread(8) = {s}, entered {calls} times");

int Pairs(int n) { calls++; return n <= 1 ? 1 : Pairs(n - 1) + Pairs(n - 2); }
int Halve(int n) { calls++; return n == 0 ? 0 : 1 + Halve(n / 2); }
int Spread(int n) { calls++; return n <= 1 ? 1 : Spread(n / 2) + Spread(n / 2); }

Pairs(5) = 8, entered 15 times
Halve(40) = 6, entered 7 times
Spread(8) = 8, entered 15 times


`Pairs(5)` returns 8 and is entered 15 times: the Fibonacci shape, where the call count grows about as fast as the value. `Halve(40)` returns 6, the number of halvings to reach 0, and is entered 7 times, one more than it returns. `Spread(8)` returns 8 and is entered 15 times: two calls on a half-size problem produce a full binary tree of calls with n leaves, 2n - 1 calls in all. Notice that `Spread` makes the identical call twice; storing the first result would halve its work, which is the observation Chapter 25 turns into memoization.

```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. A method prints a linked chain in reverse by recursing first and printing after the call. What does the call stack hold at the moment the first line prints?

```{dropdown} Answer
One frame per node, all of them: the deepest call prints first, and every earlier call is still suspended waiting to print its own node afterward. Backward flow costs stack space proportional to the length of the chain, which is why a long chain would be reversed with a loop and a stack of values instead.
```

2. Why is `Power(b, e) = b * Power(b, e - 1)` linear in e while `b * Power(b, e - 1) * b` is wrong, and how could a halving strategy make it logarithmic?

```{dropdown} Answer
The first multiplies by b exactly once per level, e levels. The second multiplies twice per level and computes b to the 2e. A halving version computes Power(b, e / 2) once, squares it, and multiplies by b when e is odd, so the depth is log e; computing Power(b, e / 2) twice instead would be correct but linear again, like Spread.
```

3. The Towers of Hanoi recurrence M(n) = 2M(n - 1) + 1 and merge sort's T(n) = 2T(n / 2) + n both make two calls. Why is one exponential and the other n log n?

```{dropdown} Answer
The size of the subproblems. Smaller by one gives n levels with the call count doubling at each, 2 to the n in total. Half size gives log n levels, and the work at each level adds up to n. Two calls are expensive only when they do not shrink the problem quickly.
```

4. In the find-the-error exercise none of the defects caused endless recursion. What single check would have caught all three before running?

```{dropdown} Answer
Evaluating the method by hand on the smallest input that reaches the base case: 0 for CountDigits, e = 1 for Power, a one-element array for Largest. Each base case is wrong or one step off, and the smallest input exposes it in one line of arithmetic.
```

```{index} Halving Strategies
```

## Halving Strategies




```{index} Halving Strategies
```

Some algorithms solve only one half after dividing. Binary search is the classic example: compare with the middle value, then search the half where the target could still be.

In [ ]:
using System;
using System.Linq;

int[] sorted={2,4,6,8,10,12,14};
foreach(int target in new[]{2,10,14,5})Console.WriteLine($"target={target}, index={Search(sorted,target)}");
Console.WriteLine($"empty={Search(Array.Empty<int>(),5)}");
int Search(int[] values,int target)
{
 ArgumentNullException.ThrowIfNull(values);
 return Find(0,values.Length);
 int Find(int start,int end)
 {
  if(start==end)return -1;
  int middle=start+(end-start)/2;
  if(values[middle]==target)return middle;
  return target<values[middle]?Find(start,middle):Find(middle+1,end);
 }
}

target=2, index=0
target=10, index=4
target=14, index=6
target=5, index=-1
empty=-1


Precondition: input is sorted in ascending order and is not modified during the search. The result is any matching index, or −1 when absent; duplicates do not promise a first match. The candidate interval is [start,end). Sortedness justifies discarding the impossible side. Excluding middle from the right interval prevents repeating a one-item range. Each surviving range is smaller; an empty range stops recursion. Only one child is explored, giving T(n)=T(floor(n/2))+Θ(1) as a size model, Θ(log n) worst-case time/stack, and Θ(1) best-case time. Checking sortedness on every call would add linear work.

**Common mistake.** Recursing on `Find(middle, end)` after learning that the target is larger than `values[middle]`. A one-element range then calls itself with the same bounds forever; excluding the middle on the right side is what guarantees the range shrinks.


```{rubric} Footnotes
```


[^2201_recursion-1]: Tail-call optimization is not something introductory C# programmers should rely on. Prefer clear loops for simple linear repetition when stack depth could become large.
[^2201_recursion-2]: A recursive method can have more than one base case. The important point is that every path of recursive calls eventually reaches one of them.


[^development-reading]: Further reading: [Nakov and colleagues, Methods](https://introprogramming.info/english-intro-csharp-book/read-online/chapter-9-methods/). The worked business scenarios and guided reasoning here are original teaching examples.
